# 03 — Exploratory Data Analysis (EDA)
**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Step:** Visualise price/volume dynamics, return distributions, and feature correlations on the processed data.

> All plots are saved to `reports/data_preparation/`.

## 1. Imports & Configuration

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

PROCESSED_DIR = os.path.join("data", "processed")
REPORT_DIR    = os.path.join("reports", "data_preparation")
os.makedirs(REPORT_DIR, exist_ok=True)

SYMBOLS = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "ITC"]
COLORS  = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
STYLE   = dict(linewidth=1.2)

## 2. Load Processed Datasets

In [ ]:
dfs = {}
for sym in SYMBOLS:
    path = os.path.join(PROCESSED_DIR, f"{sym}_processed.csv")
    df = pd.read_csv(path, parse_dates=["Date"])
    dfs[sym] = df

print("Loaded processed datasets:")
for sym, df in dfs.items():
    print(f"  {sym:<10} {df.shape[0]:>5} rows × {df.shape[1]:>2} cols  "
          f"[{df['Date'].min().date()} → {df['Date'].max().date()}]")

## 3. Closing Price History

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for (sym, df), col in zip(dfs.items(), COLORS):
    ax.plot(df["Date"], df["Close"], label=sym, color=col, **STYLE)
ax.set_title("NSE Stocks — Closing Price (5 Years)", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Close Price (INR)")
ax.legend(ncol=5, fontsize=9)
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "eda_close_prices.png"), dpi=150)
plt.show()
print("Saved: eda_close_prices.png")

## 4. Daily Volume History

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)
for ax, (sym, df), col in zip(axes, dfs.items(), COLORS):
    ax.bar(df["Date"], df["Volume"], color=col, alpha=0.7, width=1.5)
    ax.set_ylabel(sym, fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e7:.1f}Cr"))
    ax.grid(True, linestyle=":", alpha=0.4)
axes[0].set_title("Daily Volume (5 Years)", fontsize=12, fontweight="bold")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "eda_volume.png"), dpi=150)
plt.show()
print("Saved: eda_volume.png")

## 5. Daily Return Distributions

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
for ax, (sym, df), col in zip(axes, dfs.items(), COLORS):
    ax.hist(df["Daily_Return"].dropna(), bins=60, color=col, alpha=0.8, edgecolor="none")
    ax.axvline(0, color="black", lw=1, linestyle="--")
    ax.set_title(sym, fontsize=10)
    ax.set_xlabel("Daily Return")
    ax.grid(True, linestyle=":", alpha=0.4)
axes[0].set_ylabel("Frequency")
fig.suptitle("Daily Return Distributions", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "eda_return_distributions.png"), dpi=150)
plt.show()
print("Saved: eda_return_distributions.png")

## 6. Rolling 20-Day Volatility

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for (sym, df), col in zip(dfs.items(), COLORS):
    ax.plot(df["Date"], df["Rolling_Vol_20"], label=sym, color=col, **STYLE)
ax.set_title("Rolling 20-Day Volatility (σ of Daily Returns)", fontsize=12, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Volatility")
ax.legend(ncol=5, fontsize=9)
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "eda_rolling_volatility.png"), dpi=150)
plt.show()
print("Saved: eda_rolling_volatility.png")

## 7. RSI-14 Over Time

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(13, 10), sharex=True)
for ax, (sym, df), col in zip(axes, dfs.items(), COLORS):
    ax.plot(df["Date"], df["RSI_14"], color=col, **STYLE)
    ax.axhline(70, color="red",   linestyle="--", lw=0.8, alpha=0.6)
    ax.axhline(30, color="green", linestyle="--", lw=0.8, alpha=0.6)
    ax.set_ylim(0, 100)
    ax.set_ylabel(sym, fontsize=9)
    ax.grid(True, linestyle=":", alpha=0.4)
axes[0].set_title("RSI-14 (Wilder) — Overbought (>70) / Oversold (<30)", fontsize=11, fontweight="bold")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "eda_rsi.png"), dpi=150)
plt.show()
print("Saved: eda_rsi.png")

## 8. Feature Correlation Heatmap

In [ ]:
STATE_FEATURES = [
    "Daily_Return", "Log_Return", "HL_Pct", "Rolling_Vol_20",
    "SMA_Ratio_10", "SMA_Ratio_20", "SMA_Ratio_50",
    "EMA_Ratio_12", "EMA_Ratio_26", "RSI_14_Norm",
    "MACD_Line_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm", "Volume_Ratio_20",
]

all_data = pd.concat([dfs[sym][STATE_FEATURES] for sym in SYMBOLS], ignore_index=True)
corr = all_data.corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03)
ax.set_xticks(range(len(STATE_FEATURES)))
ax.set_yticks(range(len(STATE_FEATURES)))
ax.set_xticklabels(STATE_FEATURES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(STATE_FEATURES, fontsize=8)
ax.set_title("Feature Correlation Matrix (All Stocks Combined)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "eda_feature_correlation.png"), dpi=150)
plt.show()
print("Saved: eda_feature_correlation.png")

## 9. Descriptive Statistics

In [ ]:
all_stats = pd.concat([dfs[sym][STATE_FEATURES] for sym in SYMBOLS], ignore_index=True)
stats = all_stats.describe().T
stats["skew"]  = all_stats.skew()
stats["kurt"]  = all_stats.kurt()
print("Descriptive Statistics (all stocks combined):")
print(stats[["mean", "std", "min", "50%", "max", "skew", "kurt"]].round(4).to_string())